# 🧪 Lakehouse Engine: Databricks Automated End-to-End Verification Suite

Comprehensive in-cluster validation suite covering:
1. **High-Volume Synthetic Data Generation** (Controlled bad data, duplicates, PII, nested JSON)
2. **Declarative Pipeline Ingestion & Table Provisioning** (`run_pipeline`)
3. **Delta Lake Liquid Clustering** (`CLUSTER BY`) & Table Properties
4. **In-Batch Deduplication & Idempotent Upsert**
5. **Declarative DLQ Quarantine & Circuit Breaker Expectations**
6. **PII Segregation & Cryptographic Hash Masking**
7. **Native Binary `VARIANT` Rescued Data Querying**
8. **Pruned vs. Unpruned MERGE Performance Benchmark**
9. **Idempotent Historical Backfill**
10. **FinOps DBU & Cost Attribution Report**

### Step 1: Configure Parameters & Widgets

In [0]:
dbutils.widgets.text("catalog_name", "main", "1. Target Catalog")
dbutils.widgets.dropdown("scale_factor", "10000", ["1000", "10000", "100000", "1000000"], "2. Ingestion Scale (Rows)")
dbutils.widgets.dropdown("inject_chaos", "true", ["true", "false"], "3. Inject Chaos / Edge Cases")

catalog_name = dbutils.widgets.get("catalog_name")
scale_factor = int(dbutils.widgets.get("scale_factor"))
inject_chaos = dbutils.widgets.get("inject_chaos").lower() == "true"

print(f"Target Catalog: '{catalog_name}' | Scale Factor: {scale_factor:,} rows | Chaos Injection: {inject_chaos}")

Target Catalog: 'lakehouse_test' | Scale Factor: 10,000 rows | Chaos Injection: True


### Step 2: Install Engine Package

In [0]:
%pip install /Volumes/lakeshouse_engine/default/library/lakehouse_engine-0.2.0-py3-none-any.whl --force-reinstall

Looking in indexes: [REDACTED]
Processing /Volumes/lakeshouse_engine/default/library/lakehouse_engine-0.2.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.1/775.1 kB 37.1 MB/s  0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1991d72c-bf1c-4fe4-a27f-2a6e07955e4e
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.3
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1991d72c-bf1c-4fe4-a27f-2a6e07955e4e
    Can't uninstall 'PyYAML'. No files were found to uninstall.
  Attempting uninstall: click
    

### Step 3: Setup Test Schemas & Volumes

In [0]:
import time
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType, 
    TimestampType, DoubleType, IntegerType
)

test_schema = f"{catalog_name}.lakehouse_engine_test"
landing_schema = f"{catalog_name}.lakehouse_engine_landing"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {test_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {test_schema}.checkpoints")

print(f"✅ Test Schemas initialized: '{landing_schema}' and '{test_schema}'")

✅ Test Schemas initialized: 'lakehouse_test.lakehouse_engine_landing' and 'lakehouse_test.lakehouse_engine_test'


### Step 4: High-Volume Synthetic Data Injection
Generates structured stream events with controlled duplicates, NULL primary keys, and edge cases.

In [0]:
print(f"⚡ Generating {scale_factor:,} synthetic streaming records...")
t0 = time.time()

num_distinct = int(scale_factor * 0.85)
base_df = spark.range(0, scale_factor).withColumn(
    "id_num", (F.col("id") % num_distinct)
)

landing_df = base_df.select(
    F.when((F.col("id") % 100 == 0) & (F.lit(inject_chaos)), None)
     .otherwise(F.concat(F.lit("sub_"), F.col("id_num"))).alias("subscriptionId"),
    
    F.struct(
        F.when(F.col("id") % 3 == 0, F.lit("ENTERPRISE"))
         .when(F.col("id") % 3 == 1, F.lit("PRO"))
         .otherwise(F.lit("STARTER")).alias("tier"),
        F.lit("ANNUAL").alias("billingInterval"),
        ((F.col("id") % 500 + 1) * 1000).cast(LongType()).alias("amountCents")
    ).alias("plan"),
    
    F.lit("USD").alias("currency"),
    
    F.when((F.col("id") % 20 == 0) & (F.lit(inject_chaos)), F.lit("CORRUPTED_STATUS"))
     .when(F.col("id") % 2 == 0, F.lit("ACTIVE"))
     .otherwise(F.lit("CANCELLED")).alias("status"),
    
    (F.col("id") % 4 == 0).alias("isTrial"),
    F.lit(True).alias("autoRenew"),
    (F.col("id") % 5).cast(IntegerType()).alias("renewalCount"),
    
    F.from_unixtime(F.unix_timestamp() - (F.col("id") % 30) * 86400).cast("timestamp").alias("startedOn"),
    F.from_unixtime(F.unix_timestamp() - (F.col("id") % 10) * 3600).cast("timestamp").alias("updatedOn"),
    
    F.concat(F.lit("user_"), F.col("id_num"), F.lit("@example.com")).alias("email"),
    F.concat(F.lit("Customer "), F.col("id_num")).alias("customerName"),
    F.lit("192.168.1.1").alias("ipAddress"),
    
    F.to_json(F.struct(
        F.lit("MacBook Pro").alias("model"),
        F.lit("macOS 15.1").alias("os"),
        F.lit("Chrome 130").alias("browser"),
        F.lit("v2.4.0").alias("clientAppVersion")
    )).alias("deviceContext")
)

landing_table_fqdn = f"{landing_schema}.stream_subscription_events"
landing_df.write.format("delta").mode("overwrite").saveAsTable(landing_table_fqdn)

gen_duration = time.time() - t0
row_count = spark.table(landing_table_fqdn).count()
print(f"✅ Created Landing Table '{landing_table_fqdn}' with {row_count:,} rows in {gen_duration:.2f}s")

⚡ Generating 10,000 synthetic streaming records...
✅ Created Landing Table 'lakehouse_test.lakehouse_engine_landing.stream_subscription_events' with 10,000 rows in 3.95s


### Step 5: Define Pipeline Configuration & Declarative Model

In [0]:
from lakehouse_engine.core.config_loader import load_pipeline_config
from lakehouse_engine.core.guardrails import lint_pipeline_config

test_yaml = f"""
version: "1.0"
entity_name: "e2e_verification_subscriptions"
description: "End-to-End Databricks Verification Suite"

guardrails:
  enforce_primary_key_order_by: true
  allow_unbounded_merge: true
  pii_leakage_detection: "strict"

contract:
  schema_drift_policy: "fail_on_incompatible"
  max_quarantine_rate_percent: 25.0
  quarantine_action: "fail_batch"
  enforce_not_null_pk: true

source:
  catalog: "{catalog_name}"
  schema: "lakehouse_engine_landing"
  table: "stream_subscription_events"
  stream_filter_field: null

runtime:
  trigger_type: "AvailableNow"
  max_mbytes_per_trigger: 500
  checkpoint_location: "/Volumes/{catalog_name}/lakehouse_engine_test/checkpoints/e2e_verify"
  audit_table: "{catalog_name}.lakehouse_engine_test.dim_batch_audit"

gold:
  catalog: "{catalog_name}"
  schema: "lakehouse_engine_test"
  table: "dim_subscriptions"
  table_comment: "Certified Gold dimension for customer subscriptions."
  table_type: "dimension"
  write_mode: "merge"
  deduplication_mode: "in_batch"
  unique_keys: ["subscription_id"]
  cluster_by: ["started_at"]
  order_by_col: "updated_at"
  prune_column: "started_at"
  enable_dlq: true
  dlq_table: "{catalog_name}.lakehouse_engine_test.dim_subscriptions_dlq"
  track_rescued_data: true
  tags:
    tier: "gold"
    lifecycle: "production_verified"
  expectations:
    - name: "valid_subscription_status"
      expr: "status IN ('ACTIVE', 'CANCELLED', 'PAST_DUE')"
      action: "quarantine"
    - name: "positive_mrr"
      expr: "mrr_amount >= 0.0"
      action: "fail_batch"
  fields:
    - name: "subscription_id"
      expr: "subscriptionId"
      type: "string"
      is_primary_key: true
      description: "Unique surrogate key for subscription."
    - name: "plan_tier"
      expr: "plan.tier"
      type: "string"
    - name: "billing_interval"
      expr: "plan.billingInterval"
      type: "string"
    - name: "mrr_amount"
      expr: "cast(plan.amountCents as double) / 100.0"
      type: "double"
    - name: "currency"
      expr: "currency"
      type: "string"
    - name: "status"
      expr: "status"
      type: "string"
    - name: "started_at"
      expr: "startedOn"
      type: "timestamp"
    - name: "updated_at"
      expr: "updatedOn"
      type: "timestamp"
    - name: "customer_email_hash"
      expr: "xxhash64(email)"
      type: "bigint"
    - name: "client_headers"
      expr: "deviceContext"
      type: "variant"

pii_splits:
  - name: "subscriber_profiles"
    target_catalog: "{catalog_name}"
    target_schema: "lakehouse_engine_test"
    target_table: "dim_subscriber_profiles_pii"
    table_comment: "Restricted subscriber identity and PII profile dimension"
    unique_keys: ["customer_email_hash"]
    foreign_key_expr: "xxhash64(email)"
    foreign_key_name: "customer_email_hash"
    cluster_by: ["created_at"]
    fields:
      - name: "email"
        expr: "email"
        type: "string"
      - name: "customer_name"
        expr: "customerName"
        type: "string"
      - name: "ip_address"
        expr: "ipAddress"
        type: "string"
      - name: "created_at"
        expr: "startedOn"
        type: "timestamp"
"""

config = load_pipeline_config(test_yaml)
report = lint_pipeline_config(config)

print("🛡️ Pre-Flight Guardrails Check:")
print(report.format_console_report())
assert report.is_valid, "Configuration failed guardrails!"

🛡️ Pre-Flight Guardrails Check:
 🛡️ LAKEHOUSE ENGINE CONFIG GUARDRAILS: e2e_verification_subscriptions [✅ PASS]
 File:     Dynamic Configuration
 Errors:   0   |   Warnings: 0
--------------------------------------------------------------------------------
✨ All static safety and architectural guardrails passed perfectly!


### Step 6: Execute Streaming Ingestion (`run_pipeline`)

In [0]:
from lakehouse_engine.core.streaming import run_pipeline

print("🚀 Executing Streaming Ingestion Pipeline...")
t_stream_start = time.time()

# Clean previous checkpoint to guarantee deterministic test run
dbutils.fs.rm(config.runtime.checkpoint_location, recurse=True)

run_pipeline(spark, config)

batch_duration = time.time() - t_stream_start
print(f"✅ Ingestion completed in {batch_duration:.2f} seconds!")

🚀 Executing Streaming Ingestion Pipeline...
✅ Ingestion completed in 73.08 seconds!


### Step 7: Verify Liquid Clustering & Delta Table Properties

In [0]:
gold_fqdn = config.gold.fqdn
dlq_fqdn = config.gold.dlq_fqdn

detail_row = spark.sql(f"DESCRIBE DETAIL {gold_fqdn}").collect()[0]
clustering_cols = detail_row["clusteringColumns"]
print(f"✅ Target Table '{gold_fqdn}' Liquid Clustering Columns: {clustering_cols}")

assert clustering_cols is not None and len(clustering_cols) > 0, "Liquid clustering not configured!"

✅ Target Table 'lakehouse_test.lakehouse_engine_test.dim_subscriptions' Liquid Clustering Columns: ['started_at']


### Step 8: Comprehensive Verification & Test Assertions

In [0]:
results = []

def record_test(name, passed, details=""):
    status_icon = "✅ PASS" if passed else "❌ FAIL"
    results.append({"Test": name, "Status": status_icon, "Details": details})
    print(f"{status_icon} - {name} | {details}")

# Test 1: In-Batch Deduplication & Gold Row Count
gold_count = spark.table(gold_fqdn).count()
distinct_gold_pk = spark.table(gold_fqdn).select("subscription_id").distinct().count()
record_test(
    "In-Batch Deduplication",
    gold_count == distinct_gold_pk and gold_count > 0,
    f"Gold rows: {gold_count:,} | Distinct PKs: {distinct_gold_pk:,}"
)

# Test 2: Dead-Letter Queue & Expectation Quarantine
dlq_count = spark.table(dlq_fqdn).count()
if inject_chaos:
    record_test(
        "DLQ Quarantine Routing",
        dlq_count > 0,
        f"Quarantined {dlq_count:,} malformed / bad expectation rows"
    )
else:
    record_test("DLQ Zero Leakage", dlq_count == 0, f"DLQ rows: {dlq_count}")

# Test 3: PII Segregation & Deterministic Hashing
pii_fqdn = config.pii_splits[0].fqdn
pii_count = spark.table(pii_fqdn).count()
record_test(
    "PII Isolation Table",
    pii_count > 0,
    f"PII table '{pii_fqdn}' populated with {pii_count:,} records"
)

# Test 4: Native Binary VARIANT Querying
variant_sample = spark.sql(f"SELECT client_headers:os::string AS os_name FROM {gold_fqdn} LIMIT 5").collect()
record_test(
    "Photon Binary VARIANT Decoding",
    len(variant_sample) > 0 and variant_sample[0]["os_name"] == "macOS 15.1",
    f"Extracted variant value: '{variant_sample[0]['os_name']}'"
)

# Test 5: Audit Telemetry Table
audit_fqdn = config.runtime.audit_table
audit_df = spark.table(audit_fqdn).orderBy(F.col("batch_timestamp").desc())
audit_rows = audit_df.count()
record_test(
    "Audit Telemetry Logging",
    audit_rows > 0,
    f"Logged {audit_rows} batch audit records to '{audit_fqdn}'"
)

✅ PASS - In-Batch Deduplication | Gold rows: 8,075 | Distinct PKs: 8,075
✅ PASS - DLQ Quarantine Routing | Quarantined 500 malformed / bad expectation rows
✅ PASS - PII Isolation Table | PII table 'lakehouse_test.lakehouse_engine_test.dim_subscriber_profiles_pii' populated with 8,500 records
❌ FAIL - Photon Binary VARIANT Decoding | Extracted variant value: 'None'
✅ PASS - Audit Telemetry Logging | Logged 1 batch audit records to 'lakehouse_test.lakehouse_engine_test.dim_batch_audit'


### Step 9: Benchmark Pruned vs. Unpruned MERGE Performance

In [0]:
print("⏱️ Benchmarking Pruned vs. Unpruned Delta MERGE Performance...")

update_df = landing_df.limit(min(5000, scale_factor)).withColumn("plan.amountCents", F.lit(999900))

from lakehouse_engine.core.transformer import transform_entity
from lakehouse_engine.core.delta_ops import merge_batch_pruned, merge_batch_unpruned

gold_update_df = transform_entity(update_df, config.gold, config.source.payload_column, config.source.payload_type)

t_start = time.time()
merge_batch_pruned(
    spark=spark,
    target_table_fqdn=gold_fqdn,
    source_df=gold_update_df,
    unique_keys=config.gold.unique_keys,
    prune_col=config.gold.prune_column,
    order_by_col=config.gold.order_by_col
)
pruned_duration = time.time() - t_start

t_start = time.time()
merge_batch_unpruned(
    spark=spark,
    target_table_fqdn=gold_fqdn,
    source_df=gold_update_df,
    unique_keys=config.gold.unique_keys,
    order_by_col=config.gold.order_by_col
)
unpruned_duration = time.time() - t_start

speedup = unpruned_duration / pruned_duration if pruned_duration > 0 else 1.0
print(f"Pruned Merge Duration:   {pruned_duration:.3f}s")
print(f"Unpruned Merge Duration: {unpruned_duration:.3f}s")
print(f"⚡ Pruning Speedup Factor: {speedup:.2f}x")

record_test(
    "Partition Date Pruning Latency",
    pruned_duration <= unpruned_duration,
    f"Pruned: {pruned_duration:.2f}s vs Unpruned: {unpruned_duration:.2f}s ({speedup:.1f}x speedup)"
)

⏱️ Benchmarking Pruned vs. Unpruned Delta MERGE Performance...
Pruned Merge Duration:   7.780s
Unpruned Merge Duration: 6.993s
⚡ Pruning Speedup Factor: 0.90x
❌ FAIL - Partition Date Pruning Latency | Pruned: 7.78s vs Unpruned: 6.99s (0.9x speedup)


### Step 10: Idempotent Historical Backfill Execution

In [0]:
from lakehouse_engine.core.backfill import run_backfill

print("🔄 Executing Historical Backfill for date window...")
t_backfill = time.time()

run_backfill(
    spark=spark,
    config=config,
    start_date="2026-01-01",
    end_date="2026-12-31",
    date_column="startedOn",
    reset_checkpoint=False
)

backfill_duration = time.time() - t_backfill
post_backfill_gold_count = spark.table(gold_fqdn).count()

record_test(
    "Historical Backfill Idempotency",
    post_backfill_gold_count == gold_count,
    f"Gold count preserved at {post_backfill_gold_count:,} rows (0 duplicates created) in {backfill_duration:.2f}s"
)

🔄 Executing Historical Backfill for date window...

Starting Historical Backfill for: e2e_verification_subscriptions
Source: lakehouse_test.lakehouse_engine_landing.stream_subscription_events -> Target: lakehouse_test.lakehouse_engine_test.dim_subscriptions
Date Range: 2026-01-01 to 2026-12-31

Backfill records selected for processing: 10000


Expectation 'valid_subscription_status' quarantined 400 row(s). Rule: (status IN ('ACTIVE', 'CANCELLED', 'PAST_DUE'))
Writing 500 invalid record(s) to DLQ table: lakehouse_test.lakehouse_engine_test.dim_subscriptions_dlq


Quarantined 500 malformed records to lakehouse_test.lakehouse_engine_test.dim_subscriptions_dlq
Backfill MERGE complete: 0 inserted, 3325 updated.
Backfill PII table merged: lakehouse_test.lakehouse_engine_test.dim_subscriber_profiles_pii

Backfill successfully finished in 27.19 seconds.

❌ FAIL - Historical Backfill Idempotency | Gold count preserved at 8,277 rows (0 duplicates created) in 30.24s


### Step 11: FinOps DBU Consumption & Unit Cost Report

In [0]:
print("💰 Computing FinOps Ingestion Cost Metrics...")

total_rows_processed = scale_factor + min(5000, scale_factor)
total_pipeline_seconds = batch_duration + pruned_duration + backfill_duration
hours_consumed = total_pipeline_seconds / 3600.0

dbu_rate_per_hour = 3.0
dollar_per_dbu = 0.40
total_dbu_consumed = hours_consumed * dbu_rate_per_hour
total_cost_usd = total_dbu_consumed * dollar_per_dbu
cost_per_million_rows = (total_cost_usd / total_rows_processed) * 1_000_000

print(f"Total Rows Processed:       {total_rows_processed:,}")
print(f"Total Compute Execution:    {total_pipeline_seconds:.2f} seconds ({hours_consumed:.5f} hours)")
print(f"Estimated DBU Consumption:  {total_dbu_consumed:.4f} DBUs")
print(f"Estimated Cost:             ${total_cost_usd:.4f} USD")
print(f"⚡ Ingestion Unit Cost:     ${cost_per_million_rows:.4f} per 1M rows")

record_test(
    "FinOps Unit Cost SLA",
    cost_per_million_rows <= 0.25,
    f"${cost_per_million_rows:.4f} / 1M rows (SLA: <= $0.25)"
)

💰 Computing FinOps Ingestion Cost Metrics...
Total Rows Processed:       15,000
Total Compute Execution:    111.10 seconds (0.03086 hours)
Estimated DBU Consumption:  0.0926 DBUs
Estimated Cost:             $0.0370 USD
⚡ Ingestion Unit Cost:     $2.4689 per 1M rows
❌ FAIL - FinOps Unit Cost SLA | $2.4689 / 1M rows (SLA: <= $0.25)


### Step 12: Executive Verification Summary

In [0]:
import pandas as pd
summary_df = pd.DataFrame(results)
display(summary_df)

total_tests = len(results)
passed_tests = sum(1 for r in results if "PASS" in r["Status"])
pass_rate = (passed_tests / total_tests) * 100

print("\n" + "="*80)
print(f"📊 LAKEHOUSE ENGINE DATABRICKS VERIFICATION SUMMARY")
print(f"Passed: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
print("="*80)

if passed_tests == total_tests:
    print("🏆 ALL TEST SCENARIOS PASSED! The engine is verified for production deployment.")
else:
    raise Exception(f"Validation failed: {total_tests - passed_tests} test(s) did not pass.")

Test,Status,Details
In-Batch Deduplication,✅ PASS,"Gold rows: 8,075 | Distinct PKs: 8,075"
DLQ Quarantine Routing,✅ PASS,Quarantined 500 malformed / bad expectation rows
PII Isolation Table,✅ PASS,"PII table 'lakehouse_test.lakehouse_engine_test.dim_subscriber_profiles_pii' populated with 8,500 records"
Photon Binary VARIANT Decoding,❌ FAIL,Extracted variant value: 'None'
Audit Telemetry Logging,✅ PASS,Logged 1 batch audit records to 'lakehouse_test.lakehouse_engine_test.dim_batch_audit'
Partition Date Pruning Latency,❌ FAIL,Pruned: 7.78s vs Unpruned: 6.99s (0.9x speedup)
Historical Backfill Idempotency,❌ FAIL,"Gold count preserved at 8,277 rows (0 duplicates created) in 30.24s"
FinOps Unit Cost SLA,❌ FAIL,$2.4689 / 1M rows (SLA: <= $0.25)



📊 LAKEHOUSE ENGINE DATABRICKS VERIFICATION SUMMARY
Passed: 4/8 (50.0%)


---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-7871584937506664>, line 17
     15     print("🏆 ALL TEST SCENARIOS PASSED! The engine is verified for production deployment.")
     16 else:
---> 17     raise Exception(f"Validation failed: {total_tests - passed_tests} test(s) did not pass.")

Exception: Validation failed: 4 test(s) did not pass.